# 파이썬 비동기(Asyncio) 



## 1) 비동기란?
- 동기 코드는 I/O(웹/DB/파일)를 기다리는 동안 **아무 일도 하지 못하고 멈춤**
- 비동기는 기다리는 동안 **다른 작업**을 진행 → 처리량 증가, 지연 감소

## 2) 동기 vs 비동기

In [ ]:
import time, asyncio

def sync_download(idx, delay=0.4):
    time.sleep(delay)          #기다리는 동안 CPU가 다른걸 하지 못함
    return f"sync-{idx}"

def demo_sync(n=5):
    t0 = time.perf_counter()
    out = [sync_download(i) for i in range(n)]  # 하나 끝나야 다음 시작
    print("sync elapsed:", round(time.perf_counter()-t0, 2), "s")
    return out

print("동기 결과:", demo_sync())

sync elapsed: 2.0 s
동기 결과: ['sync-0', 'sync-1', 'sync-2', 'sync-3', 'sync-4']


In [ ]:
async def async_download(idx, delay=0.4):
    await asyncio.sleep(delay) # await는 기다리는 동안 이벤트 루프에 제어권을 양보하고 다른 작업 수행
    return f"async-{idx}"

async def demo_async(n=5):
    t0 = time.perf_counter()
    # 5개 작업을 '한꺼번에' 스케줄하고 모두 끝날 때까지 대기
    out = await asyncio.gather(*[async_download(i) for i in range(n)]) #gather는 모든 작업이 끝날 때까지 기다림
    print("async elapsed:", round(time.perf_counter()-t0, 2), "s")
    return out

print("비동기 결과:", await demo_async())

async elapsed: 0.4 s
비동기 결과: ['async-0', 'async-1', 'async-2', 'async-3', 'async-4']


## 3) 이벤트 루프, 코루틴, awaitable
- **이벤트 루프**: 비동기 작업을 스케줄·실행하는 런타임
- **코루틴**: `async def`로 정의된 함수가 **호출되면** 생기는 객체(즉시 실행 아님)
- **awaitable**: `await` 가능한 대상(코루틴, Task 등)

In [26]:
import asyncio

async def tiny_loop():
    print("tiny_loop 시작")
    await asyncio.sleep(0.1)
    print("tiny_loop 완료")
    return "done"

coro = tiny_loop()                  # 코루틴 객체 생성 (아직 실행되지 않음)

print('---------------------------------')

result = await coro            # 여기서 실제 실행됨
print("결과:", result)


---------------------------------
tiny_loop 시작
tiny_loop 완료
결과: done


## 4) 태스크(Task)란?
- `asyncio.create_task(coro)`로 코루틴을 **백그라운드**에서 진행하도록 스케줄
- 태스크는 "이미 실행 중"일 수 있으므로, 필요할 때만 `await`로 **결과를 수거**

In [23]:

import asyncio, time

async def io_like(name, delay=0.3):
    await asyncio.sleep(delay)
    print(f"{name} 완료 (delay={delay})")
    return name

async def demo_tasks():
    t0 = time.perf_counter()
    # 동시에 세 작업을 시작
    t1 = asyncio.create_task(io_like("A", 0.5))
    t2 = asyncio.create_task(io_like("B", 0.2))
    t3 = asyncio.create_task(io_like("C", 0.3))

    r1 = await t1
    r2 = await t2
    r3 = await t3

    print("결과:", r1, r2, r3, "| 전체 시간은", round(time.perf_counter()-t0, 2), "s 소요")

await demo_tasks()


B 완료 (delay=0.2)
C 완료 (delay=0.3)
A 완료 (delay=0.5)
결과: A B C | 전체 시간은 0.5 s 소요


## 5) 여러 작업을 한 번에 기다리기: `asyncio.gather`
- 인자로 준 여러 awaitable을 **모두 완료**할 때까지 기다림
- 기본값: 하나라도 예외가 나면 **즉시 전파** (전체가 실패할 수 있음)
- 예외를 값처럼 다루고 싶다면 `return_exceptions=True`

In [21]:

import asyncio, random

async def maybe_fail(i):
    await asyncio.sleep(0.2)
    if i % 4 == 0:
        raise RuntimeError(f"실패 했습니다!! (@.@) {i}")
    return i

async def demo_gather():
    try:
        res = await asyncio.gather(*[maybe_fail(i) for i in range(1, 7)])
        print("모두 성공:", res)
    except Exception as e:
        print("무언가 하나가 실패 했습니다.! ", type(e).__name__, e) #<-- 하나라도 실패 하면 끝
        
    print('--------------------------------')

    # 예외를 값처럼 받고 싶을 때
    res2 = await asyncio.gather(*[maybe_fail(i) for i in range(1, 7)], return_exceptions=True)
    print("return_exceptions=True 결과:")
    for i, r in enumerate(res2, start=1):
        print(f"  {i}: {repr(r)}")

await demo_gather()


무언가 하나가 실패 했습니다.!  RuntimeError 실패 했습니다!! (@.@) 4
--------------------------------
return_exceptions=True 결과:
  1: 1
  2: 2
  3: 3
  4: RuntimeError('실패 했습니다!! (@.@) 4')
  5: 5
  6: 6


## 6) 먼저 끝나는 것부터 처리: `asyncio.as_completed`
- 대량 작업에서 "완료되는 즉시" 결과를 소비할 때 유용

In [15]:

import asyncio, random, time

async def work(i):
    d = random.uniform(0.1, 0.6)
    await asyncio.sleep(d)
    return (i, round(d, 2))

async def demo_as_completed():
    tasks = [asyncio.create_task(work(i)) for i in range(10)]
    t0 = time.perf_counter()
    out = []
    for fut in asyncio.as_completed(tasks):
        out.append(await fut)  # 먼저 끝난 순서대로 결과 처리
    print("완료 순서:", out)
    print("elapsed:", round(time.perf_counter()-t0, 2), "s")

await demo_as_completed()


완료 순서: [(1, 0.13), (6, 0.23), (7, 0.33), (0, 0.34), (2, 0.35), (4, 0.36), (5, 0.48), (8, 0.53), (9, 0.56), (3, 0.59)]
elapsed: 0.59 s


## 7) 타임아웃: `asyncio.wait_for`
- 외부 I/O 지연으로 **무한 대기**를 방지
- 타임아웃 발생 시 `asyncio.TimeoutError`

In [28]:

import asyncio

async def slow(): # 실제 이 작업은 1초가 걸림
    await asyncio.sleep(1.0) 
    return "OK after 1s"

async def demo_timeout():
    try:
        print(await asyncio.wait_for(slow(), timeout=0.3)) # 타임 아웃이 0.3초 이상 걸리면 예외 발생
    except asyncio.TimeoutError:
        print("타임아웃 발생! (0.3s가 넘게 지연됨)")

await demo_timeout()


타임아웃 발생! (0.3s가 넘게 지연됨)


## 8) 취소(Cancellation)와 정리(Cleanup)
- `task.cancel()` 호출 시 코루틴 내부에서 `CancelledError`가 발생
- 파일/락/네트워크 자원 정리가 필요하면 `try/except/finally`로 마무리 코드 작성

In [ ]:

import asyncio

async def cancellable():
    try:
        for _ in range(5):
            await asyncio.sleep(0.2)  # 긴 작업의 일부라고 가정
        return "완료"
    except asyncio.CancelledError:
        print("취소 신호 수신 → 정리 중...")
        raise
    finally:
        pass

async def demo_cancel():
    t = asyncio.create_task(cancellable())   #<-- 취소 될지도 모르는 작업
    await asyncio.sleep(0.5)  # 조금 돌리다가
    t.cancel()                # 취소하면
    try:
        await t                 # 여기에서 에러가 발생하면 취소된 것임
    except asyncio.CancelledError:
        print("작업이 정상적으로 취소되었습니다.")

await demo_cancel()

취소 신호 수신 → 정리 중...
작업이 정상적으로 취소되었습니다.


## 9) 세마포어로 **동시에 몇 개까지** 돌릴지 제한하기
- 외부 API/DB가 동시 연결 수를 제한할 때 필수
- `async with sem:` 블록이 끝나면 자동으로 반납되므로 안전

In [11]:
import asyncio, random, time

sem = asyncio.Semaphore(3)  # 동시에 3개만 실행 허용

async def limited_job(i):
    async with sem:  
        await asyncio.sleep(random.uniform(0.2, 0.5))
        return i

async def demo_semaphore():
    tic = time.perf_counter()
    # 동기 였다면 10초 이상 걸림
    # 완전 비동기 였다면 0.5초 정도 걸림
    # 최대 3개만 동시에 실행되므로 최악의 경우 1.5초 정도 걸림
    res = await asyncio.gather(*[limited_job(i) for i in range(10)])
    toc = time.perf_counter()
    print("결과:", res, "| elapsed:", round(toc-tic, 2), "s")

await demo_semaphore()

결과: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9] | elapsed: 1.42 s


## 10) 미니 벤치마크: 동기 vs 비동기 (재확인)
- 여러 I/O를 **동시에** 대기하면 총 시간은 거의 "최장 지연"에 가까워집니다.

In [8]:
import time, asyncio

def sync_batch(n=20, d=0.2):
    tic = time.perf_counter()
    for _ in range(n):
        time.sleep(d)
    toc = time.perf_counter()
    return toc-tic

async def async_batch(n=20, d=0.2):
    async def one():
        await asyncio.sleep(d)
    tic = time.perf_counter()
    await asyncio.gather(*[one() for _ in range(n)])
    toc = time.perf_counter()
    return toc-tic

s = sync_batch()
a = await async_batch()
print(f"sync : {s:.2f}s")
print(f"async: {a:.2f}s")

sync : 4.00s
async: 0.20s
